A lo largo de las sesiones de laboratorio, se ha llegado a alcanzar con el clasificador de **Regresión Logística (LR)** un acierto del 92.6% en la tarea de MNIST (sin ingeniería de características). Con el objetivo de intentar mejorar esta tasa de acierto, completa el siguiente cuaderno para realizar un experimento con LR en el que se varie la dimensionalidad de los datos mediante PCA, y se haga una exploración de los valores del hiperparámetro de tolerancia del criterio de parada (`tol`). Completa la variable dni con las tres últimas cifras de tu DNI sin letra para ser usada como semilla de inicialización en la exploración de hiperparámetros. Tras realizar el experimento, comenta cuáles són los valores óptimos que has obtenido y cómo los has detectado.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

In [2]:
dni=937 # Completar

In [3]:
X, y = fetch_openml(data_id=554, return_X_y=True, as_frame=False, parser='liac-arff'); X /= 255.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, shuffle=False)

In [5]:
import time; start = time.time()
max_K = np.min(X_train.shape); pca = PCA(n_components=max_K).fit(X_train)
X_train = pca.transform(X_train); X_test = pca.transform(X_test) # Completar
Ks = np.array([100, 200, 500, max_K]) # Completar
clf = LogisticRegression()
G = {'tol': [1e-6, 1e-4, 1e-2]} # Completar
splitter = ShuffleSplit(n_splits=1, test_size=0.1, random_state=dni)
RS = RandomizedSearchCV(clf, G, n_iter=3, scoring='accuracy', n_jobs=1, cv=splitter, pre_dispatch=32, random_state=dni)
for _, k in enumerate(reversed(np.sort(Ks))):
    Z_train = X_train[:,:k]
    Z_test = X_test[:,:k]
    
    acc = RS.fit(Z_train, y_train).score(Z_test, y_test) # Completar
    print(f'Precisión: {acc:.2%} con {RS.best_params_} y K = {k}')
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 92.62% con {'tol': 1e-06} y K = 784
Precisión: 92.59% con {'tol': 1e-06} y K = 500
Precisión: 92.61% con {'tol': 0.0001} y K = 200
Precisión: 92.18% con {'tol': 0.0001} y K = 100
Tiempo (hh:mm:ss): 00:02:03


A partir de los resultados obtenidos, ¿Qué combinación del valor de K y el valor del hiperparámetro tol eligirías?

RESPUESTA:

En este caso, podemos considerar trabajar con `K=200`, puesto que supone una pérdida de rendimiento muy pequeña con respecto a usar las muestras sin transformar (92.61 frente a 92.62).
Sabiendo que vamos a trabajar con `k=200`, `tol=1e-4` es un mínimo local, ya que hemos considerado `tol={1e-2, 1e-4, 1e-6}` y el valor de `tol` con mejores resultados para `K=200` es el que aparece en el output, `1e-4`.